In [1]:
import os
import cv2
import numpy as np

# Updated Paths
image_dir = "./content/auto_dataset/valid/images"
label_dir = "./content/auto_dataset/valid/labels"
image_ext = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

# Map images
images = [f for f in os.listdir(image_dir) if os.path.splitext(f)[1].lower() in image_ext]
image_map = {os.path.splitext(f)[0]: f for f in images}
labels = [f for f in os.listdir(label_dir) if f.endswith(".txt")]

CLASS_COLORS = {
    0: (0, 255, 0), # Green Leaf
    1: (0, 0, 255), # Red Disease
}

def get_color(cls):
    if cls in CLASS_COLORS:
        return CLASS_COLORS[cls]
    np.random.seed(cls)
    return tuple(np.random.randint(0, 256, 3).tolist())

print("Showing masks... Press any key for next, 'q' to quit.\n")

for label_file in labels:
    name = os.path.splitext(label_file)[0]
    if name not in image_map:
        continue

    img_path = os.path.join(image_dir, image_map[name])
    label_path = os.path.join(label_dir, label_file)
    
    img = cv2.imread(img_path)
    if img is None: continue
    
    h, w = img.shape[:2]
    mask = np.zeros((h, w, 3), dtype=np.uint8)

    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 3: continue
            
            cls = int(float(parts[0]))
            coords = list(map(float, parts[1:]))
            
            # Reshape and scale coordinates
            points = np.array(coords).reshape((-1, 2))
            points[:, 0] *= w
            points[:, 1] *= h
            points = points.astype(np.int32)
            
            cv2.fillPoly(mask, [points], get_color(cls))

    # --- DISPLAY LOGIC ---
    # Create an overlay to see alignment
    overlay = cv2.addWeighted(img, 0.5, mask, 0.5, 0)
    
    # Combine original, mask, and overlay for one big preview
    combined = np.hstack((img, mask, overlay))
    
    # Resize window if the image is too large for your screen
    cv2.namedWindow("Preview: Original | Mask | Overlay", cv2.WINDOW_NORMAL)
    cv2.imshow("Preview: Original | Mask | Overlay", combined)

    key = cv2.waitKey(0) & 0xFF
    if key == ord('q'):
        break

cv2.destroyAllWindows()


Showing masks... Press any key for next, 'q' to quit.

